# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
We use the `mlcroissant` library to load the FAIR² dataset's metadata and records via the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset and its metadata
dataset = mlc.Dataset(croissant_url)

# Print summary description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and fields using their `@id` values as required. We will list all record sets, their `@id`s, and the fields (`@id`s and names) contained in each. This overview helps decide what data to load for further exploration.

In [ ]:
# List all available record sets with fields, referenced by their @id
record_sets = dataset.record_sets()
if not record_sets:
    print("No record sets available in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs['name']}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    - @id: {field['@id']}, name: {field.get('name', '<no name>')}")
        print()

For illustration, let's explicitly list and choose a record set to demonstrate further analysis.

**Example output (actual may differ):**
```
RecordSet @id: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#SecondPrimaryColorectalCancerPatients
  Name: SecondPrimaryColorectalCancerPatients
  Fields:
    - @id: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#Age, name: Age
    - @id: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#Sex, name: Sex
    ...
```
Use the displayed @id values in the next steps.

## 3. Data Extraction
Now, let's extract data for a chosen record set (e.g., `SecondPrimaryColorectalCancerPatients`) using its `@id` and load it into a pandas DataFrame. All references are by `@id`.

Please update `selected_record_set_id` as needed based on the above overview.

In [ ]:
# Specify the @id for the record set of interest (update if your dataset uses a different @id)
selected_record_set_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#SecondPrimaryColorectalCancerPatients'

# List all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print('Available record set @ids:')
for rsid in record_set_ids:
    print('  ', rsid)

# Extract all records for the selected record set
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f'Fields in records for {selected_record_set_id}:')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by key variables.

- **Filtering:** Keep only patients above a given age.
- **Normalizing:** Z-score normalization of a numeric field.
- **Grouping:** Compute grouped means by patient `Sex`.

All variables are referenced by their `@id` according to FAIR data practices.

In [ ]:
# Choose the proper @ids for fields based on the above output
# Example field @ids (to be replaced with actual @ids from your Data Overview):
age_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#Age'
sex_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#Sex'

# Filter: Only patients older than 50
age_threshold = 50

filtered_df = df[df[age_field_id] > age_threshold].copy()
print(f"Number of records with Age (@id={age_field_id}) > {age_threshold}: {filtered_df.shape[0]}")
print(filtered_df[[age_field_id, sex_field_id]].head())

# Normalize the Age field (z-score)
filtered_df[f"{age_field_id}_zscore"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
print(f"Normalized (z-score) values for Age (@id={age_field_id}):")
print(filtered_df[[age_field_id, f"{age_field_id}_zscore"]].head())

# Group: Mean Age by Sex
if sex_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(sex_field_id)[age_field_id].mean().reset_index()
    print(f"Mean Age by Sex (@id={sex_field_id}):")
    print(grouped)
else:
    print(f"Sex field (@id={sex_field_id}) not found in DataFrame.")

## 5. Visualization
Let's plot the distribution of patient age and visualize mean age by sex.

All plots label axes by field `@id` as per FAIR guidelines.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of Age
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[age_field_id], bins=10, kde=True)
plt.title(f'Age Distribution (@id={age_field_id})')
plt.xlabel(age_field_id)
plt.ylabel('Count')
plt.show()

# Bar plot of mean Age by Sex if grouping available
if sex_field_id in filtered_df.columns:
    plt.figure(figsize=(6,4))
    sns.barplot(x=sex_field_id, y=age_field_id, data=grouped)
    plt.title(f'Mean Age by Sex (@id={sex_field_id})')
    plt.xlabel(sex_field_id)
    plt.ylabel(f'Mean {age_field_id}')
    plt.show()

## 6. Conclusion
- We loaded a clinical oncology FAIR² dataset via Croissant and explored records by referencing all entities with their `@id`.
- Out of 77 patients, filtering for age > 50 selected a subset for analysis.
- Normalization and group analysis by sex showed distribution differences.
- All field access and references used fully-qualified `@id` fields, enabling reproducibility and interoperability.

To extend the analysis, use additional `@id` references for other record sets, fields, or incorporate FIAR2 metadata for richer exploration.